# AutoVoice: Train Severity Classifier (Triage Prioritization)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devdasrahul/autovoice/blob/main/notebooks/03_train_classifier_colab.ipynb)

This notebook trains the AutoVoice triage severity classifier on NHTSA complaint narratives.
- **Architecture:** Sublinear TF-IDF (1-2 word n-grams) + Class-Weighted Logistic Regression (`class_weight='balanced'`)
- **Proxy Severity Labels:** Derived strictly from objective indicator fields (`deaths`, `injured`, `fire`, `crash`) with zero narrative text leakage.
- **Triage Classes:** `LOW`, `MEDIUM`, `HIGH`

**Google Colab Ready:**
- Prompts to upload `complaints.parquet` if not already present.
- When training completes, confusion matrix plot and model artifacts (`model.pkl`, `vectorizer.pkl`) are **automatically downloaded** to your machine.


In [ ]:
# 1. Environment Setup
!pip install -q scikit-learn pandas numpy pyarrow matplotlib seaborn

import os, sys, json, pickle, shutil
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import train_test_split

try:
    from google.colab import files
    IN_COLAB = True
    print('🚀 Running in Google Colab.')
except ImportError:
    IN_COLAB = False
    print('💻 Running locally.')


In [ ]:
# 2. Interactive Data Upload / Loading
data_file = None
candidate_paths = [
    Path('complaints.parquet'),
    Path('data/processed/complaints.parquet'),
    Path('../data/processed/complaints.parquet'),
    Path('nhtsa_complaints_raw.csv'),
]

for cp in candidate_paths:
    if cp.exists():
        data_file = cp
        break

if data_file is None and IN_COLAB:
    print('📁 Please upload complaints.parquet or nhtsa_complaints_raw.csv:')
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(('.parquet', '.csv')):
            data_file = Path(fname)
            break

if data_file is None:
    raise FileNotFoundError('No complaint dataset found. Please upload complaints.parquet.')

print(f'✅ Loading dataset from {data_file}')
df = pd.read_parquet(data_file) if str(data_file).endswith('.parquet') else pd.read_csv(data_file)
text_col = 'narrative' if 'narrative' in df.columns else ('cdescr' if 'cdescr' in df.columns else df.columns[0])

# Derive severity labels from physical indicators if not already computed
if 'severity' not in df.columns:
    def derive_severity(row):
        deaths = int(row.get('deaths', 0) or 0)
        injured = int(row.get('injured', 0) or 0)
        fire = bool(row.get('fire', False))
        crash = bool(row.get('crash', False))
        if deaths > 0 or injured > 0 or fire:
            return 'HIGH'
        if crash:
            return 'MEDIUM'
        return 'LOW'
    df['severity'] = df.apply(derive_severity, axis=1)

print('Class distribution:\n', df['severity'].value_counts())


In [ ]:
# 3. Train / Test Split & Feature Extraction
X = df[text_col].fillna('').astype(str).tolist()
y = df['severity'].tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train samples: {len(X_train)}, Test samples: {len(X_test)}')

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    sublinear_tf=True,
    strip_accents='unicode',
    lowercase=True,
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print(f'Vocabulary size: {len(vectorizer.vocabulary_)} features')


In [ ]:
# 4. Train Class-Weighted Logistic Regression Model
clf = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    C=1.0,
    random_state=42,
)

print('Training severity classifier...')
clf.fit(X_train_vec, y_train)
print('Training complete!')


In [ ]:
# 5. Evaluation & Confusion Matrix Heatmap
output_dir = Path('classifier_artifacts')
output_dir.mkdir(parents=True, exist_ok=True)

y_pred = clf.predict(X_test_vec)
acc = accuracy_score(y_test, y_pred)
weighted_f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

print(f'Accuracy: {acc*100:.2f}%')
print(f'Weighted F1: {weighted_f1:.4f}')
print(f'Macro F1: {macro_f1:.4f}\n')
print('Classification Report:\n', classification_report(y_test, y_pred, zero_division=0))

classes = sorted(list(set(y)))
cm = confusion_matrix(y_test, y_pred, labels=classes)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_title('Triage Severity Confusion Matrix')
ax.set_xlabel('Predicted Class')
ax.set_ylabel('True Indicator Class')
plt.tight_layout()
fig.savefig(output_dir / 'confusion_matrix.png', dpi=150)
plt.show()


In [ ]:
# 6. Export Model, Vectorizer & Metrics
with open(output_dir / 'model.pkl', 'wb') as f:
    pickle.dump(clf, f)
with open(output_dir / 'vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

metrics = {
    'accuracy': round(float(acc), 4),
    'weighted_f1': round(float(weighted_f1), 4),
    'macro_f1': round(float(macro_f1), 4),
    'classes': classes,
}
(output_dir / 'clf_metrics.json').write_text(json.dumps(metrics, indent=2))
print('Saved model and metrics to:', output_dir)


In [ ]:
# 7. Interactive Triage Inference Sample
test_samples = [
    'Vehicle experienced a minor cosmetic scratch on plastic hubcap while parked.',
    'Brakes failed at 50 mph resulting in rear-end crash with guardrail.',
    'High-voltage battery caught fire on highway. Ambulance called for driver burns.',
]

for sample in test_samples:
    vec = vectorizer.transform([sample])
    pred = clf.predict(vec)[0]
    probs = {cls_name: round(float(p), 4) for cls_name, p in zip(clf.classes_, clf.predict_proba(vec)[0])}
    print(f'Text: "{sample}"')
    print(f'-> Predicted Severity: {pred}')
    print(f'-> Calibrated Distribution: {probs}\n')


In [ ]:
# 8. Automated Output Packaging & Browser Download
archive_name = 'autovoice_classifier_artifacts'
shutil.make_archive(archive_name, 'zip', output_dir)
print(f'📦 Packaged Classifier model artifacts into {archive_name}.zip')

if IN_COLAB:
    print('⬇️ Downloading model artifacts to your local machine...')
    files.download(f'{archive_name}.zip')
else:
    print(f'Archive created at: {Path(archive_name + ".zip").resolve()}')
